# 💵 Annual Gross Price Dimension (`dim_gross_price`) Lakehouse Pipeline
This notebook implements the complete end-to-end Medallion pipeline for the **Product Pricing Index**:
* **Bronze:** Ingests raw CSV pricing landing files with explicit schemas and Change Data Feed.
* **Silver:** Standardizes multi-format dates, cleanses price symbols/negatives, and broadcast-joins with product master to attach conformed `product_code`.
* **Gold:** Persists subsidiary monthly pricing `sb_dim_gross_price`, consolidates annual prices via window ranking, verifies data quality, and executes SCD Type 1 merge into enterprise parent table `dim_gross_price`.

### 📌 Step 1: Import Core PySpark, Delta Lake & Window Functions
* **Purpose:** Imports SQL functions, Delta Lake table APIs, and Window specifications required for deduplication and price ranking.
* **Logic & Transformations:** Imports `pyspark.sql.functions as F`, `DeltaTable`, and `Window`.
* **Inputs & Dependencies:** PySpark runtime and `delta-spark` package.
* **Outputs & Medallion State:** Modules `F`, `DeltaTable`, and `Window` available in session scope.

In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

### 📌 Step 2: Runtime Bootstrap & Project Utilities Execution
* **Purpose:** Configures modular Python paths and executes shared project utilities to establish environment configurations and schemas.
* **Logic & Transformations:** Resolves repository root path on `sys.path`, executes `%run ./utilities`, and initializes local compatibility context.
* **Inputs & Dependencies:** Shared Lakehouse utilities (`./utilities.py` / `utilities.ipynb`).
* **Outputs & Medallion State:** Pre-populated `spark`, `dbutils`, `display`, and global configurations in session scope.

In [2]:
# Initialize environment & Databricks compatibility (noop in Databricks)
import sys, os
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) in ["1_setup", "2_dimension_data_processing", "3_fact_dat_processing"] else current_dir
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.compat import init_notebook_context
spark, dbutils, display = init_notebook_context(globals())

# Load environment config, schemas, and utilities via relative path
%run ../1_setup/utilities


26/09/17 12:41:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 📌 Step 3: Verify Active Medallion Schema Configurations
* **Purpose:** Validates active Lakehouse schemas (`bronze`, `silver`, `gold`) for the current execution tier.
* **Logic & Transformations:** Prints schema strings to standard output for visual verification.
* **Inputs & Dependencies:** Configuration variables exported by utilities in Step 2.
* **Outputs & Medallion State:** Schema names printed to cell output.

In [3]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


### 📌 Step 4: Pipeline Parameterization via Interactive Widgets
* **Purpose:** Establishes configurable parameters (`catalog`, `data_source` = `gross_price`) and derives cloud storage URIs for the raw pricing landing zone.
* **Logic & Transformations:** Registers text widgets, resolves active catalog and dataset name, and forms S3 paths (`base_path`, `landing_path`, `processed_path`).
* **Inputs & Dependencies:** Databricks widget inputs (`catalog`: `fmcg`, `data_source`: `gross_price`).
* **Outputs & Medallion State:** Pipeline URI paths and target table names (`bronze_table`, `silver_table`, `gold_table`) registered.

In [4]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://spartsbar-2355/{data_source}/*.csv'
print(base_path)

s3://spartsbar-2355/gross_price/*.csv


### 📌 Step 5: Schema-Enforced Ingestion from AWS S3 Landing Zone
* **Purpose:** Reads landed gross price CSV files using explicit schema enforcement, capturing ingestion audit metadata.
* **Logic & Transformations:** Binds explicit `pricing_schema`, appends `current_timestamp()` as `read_timestamp`, and unpacks `_metadata.file_name` and `_metadata.file_size`.
* **Inputs & Dependencies:** Raw CSV files at `s3://spartsbar-2355/gross_price/*.csv`.
* **Outputs & Medallion State:** Raw DataFrame `df` containing source pricing records with ingestion metadata.

In [5]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(pricing_schema)  # Explicit schema avoids inferSchema overhead
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))


26/09/17 12:41:46 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3://spartsbar-2355/gross_price/*.csv.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3586)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(

[Local Spark Emulation] S3 path detected without AWS credentials. Providing mock data for: s3://spartsbar-2355/gross_price/*.csv
+----------+-----+-----------+---------+--------------+---------+---------+
|product_id|month|gross_price|_metadata|read_timestamp|file_name|file_size|
+----------+-----+-----------+---------+--------------+---------+---------+
+----------+-----+-----------+---------+--------------+---------+---------+



### 📌 Step 6: Validate Raw Ingestion Schema
* **Purpose:** Inspects field data types and nullability constraints to ensure strict alignment with enterprise ingestion standards.
* **Logic & Transformations:** Calls `df.printSchema()` to print the DataFrame structural tree.
* **Inputs & Dependencies:** Ingested DataFrame `df` from Step 5.
* **Outputs & Medallion State:** Schema definition logged to cell output.

In [6]:
# print check data type
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- _metadata: struct (nullable = false)
 |    |-- file_name: string (nullable = false)
 |    |-- file_size: long (nullable = false)
 |    |-- file_path: string (nullable = false)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### 📌 Step 7: Inspect Sample Raw Pricing Records
* **Purpose:** Visually validates the integrity and layout of landed pricing records before committing to the Delta Lake storage layer.
* **Logic & Transformations:** Limits DataFrame to 10 rows and invokes `display(df.limit(10))` for tabular presentation.
* **Inputs & Dependencies:** Raw DataFrame `df`.
* **Outputs & Medallion State:** Formatted tabular view rendered in notebook output.

In [7]:
display(df.limit(10))

+----------+-----+-----------+---------+--------------+---------+---------+
|product_id|month|gross_price|_metadata|read_timestamp|file_name|file_size|
+----------+-----+-----------+---------+--------------+---------+---------+
+----------+-----+-----------+---------+--------------+---------+---------+



### 📌 Step 8: Persist Raw Ingestion into Bronze Delta Table
* **Purpose:** Saves landed pricing data into immutable Bronze Delta Lake table with Change Data Feed enabled.
* **Logic & Transformations:** Writes `df` with `format('delta')`, sets `delta.enableChangeDataFeed = true`, and saves in `overwrite` mode to `{catalog}.bronze.gross_price`.
* **Inputs & Dependencies:** Raw DataFrame `df`.
* **Outputs & Medallion State:** Bronze Delta table `fmcg.bronze.gross_price` committed on Delta Lake storage.

In [8]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.bronze.gross_price -> bronze.gross_price


26/09/17 12:41:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### 📌 Step 9: Query Bronze Table to Initialize Silver Cleansing
* **Purpose:** Reads raw records back from Bronze Delta storage to isolate raw landing from transformation logic.
* **Logic & Transformations:** Executes `spark.sql(SELECT * FROM {catalog}.{bronze_schema}.{data_source};)` and previews 10 records.
* **Inputs & Dependencies:** Bronze Delta table `fmcg.bronze.gross_price`.
* **Outputs & Medallion State:** DataFrame `df_bronze` loaded into memory for Silver tier processing.

In [9]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.bronze.gross_price; -> SELECT * FROM bronze.gross_price;
+----------+-----+-----------+---------+--------------+---------+---------+
|product_id|month|gross_price|_metadata|read_timestamp|file_name|file_size|
+----------+-----+-----------+---------+--------------+---------+---------+
+----------+-----+-----------+---------+--------------+---------+---------+



### 📌 Step 10: Profile Distinct Raw Month Date Strings
* **Purpose:** Scans the `month` column to identify all date formats, delimiters, and string representations present in the source files.
* **Logic & Transformations:** Queries `df_bronze.select('month').distinct().show()` to list all distinct raw date strings.
* **Inputs & Dependencies:** Bronze DataFrame `df_bronze`.
* **Outputs & Medallion State:** Terminal tabular output of distinct raw month values.

In [10]:
df_bronze.select('month').distinct().show()

+-----+
|month|
+-----+
+-----+



### 📌 Step 11: Multi-Format Date Parsing & Coalesce Normalization
* **Purpose:** Standardizes multi-format date strings into native DateType using candidate format parsing.
* **Logic & Transformations:** Evaluates candidate patterns (`yyyy/MM/dd`, `dd/MM/yyyy`, `yyyy-MM-dd`, `dd-MM-yyyy`) via `F.to_date` and takes the first matching non-null date via `F.coalesce`.
* **Inputs & Dependencies:** List `date_formats` and `df_bronze` DataFrame.
* **Outputs & Medallion State:** Normalized `month` column of type `date`.

In [11]:

# 1️. Parse `month` from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

### 📌 Step 12: Verify Standardized Month Dates
* **Purpose:** Confirms that all date string variations were successfully parsed into valid ISO dates without unhandled formats.
* **Logic & Transformations:** Queries `df_silver.select('month').distinct().show()` to verify clean distinct dates.
* **Inputs & Dependencies:** Transformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Verified distinct month dates logged to output.

In [12]:
df_silver.select('month').distinct().show()

+-----+
|month|
+-----+
+-----+



### 📌 Step 13: Inspect Intermediate Price Data
* **Purpose:** Inspects sample records to examine raw `gross_price` string formatting (currencies, negative signs, text placeholders).
* **Logic & Transformations:** Calls `df_silver.show(10)` to render sample pricing rows.
* **Inputs & Dependencies:** DataFrame `df_silver`.
* **Outputs & Medallion State:** Sample records printed to cell output.

In [13]:
df_silver.show(10)

+----------+-----+-----------+---------+--------------+---------+---------+
|product_id|month|gross_price|_metadata|read_timestamp|file_name|file_size|
+----------+-----+-----------+---------+--------------+---------+---------+
+----------+-----+-----------+---------+--------------+---------+---------+



### 📌 Step 14: Sanitize Gross Price & Convert to Numeric Double
* **Purpose:** Cleans the price column by stripping currency symbols/commas, correcting accidental negative values to positive (`abs`), and nullifying non-numeric artifacts.
* **Logic & Transformations:** Strips whitespace and characters via `regexp_replace`, casts valid numeric strings to `DoubleType`, and enforces non-negative values via `F.abs()`.
* **Inputs & Dependencies:** DataFrame `df_silver`.
* **Outputs & Medallion State:** Cleaned `gross_price` column of type `double`.

In [14]:
# We are validating the gross_price column, converting only valid numeric values to double, fixing negative prices by making them positive, and replacing all non-numeric values with 0


df_silver = df_silver.withColumn(
    "gross_price",
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), 
           F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double"))
            .otherwise(F.col("gross_price").cast("double")))
    .otherwise(0)
)

### 📌 Step 15: Preview Sanitized Price Values
* **Purpose:** Verifies that price formatting and numeric conversion succeeded across sample rows.
* **Logic & Transformations:** Calls `df_silver.show(10)` to render verified pricing data.
* **Inputs & Dependencies:** Transformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Formatted pricing records printed to output.

In [15]:
df_silver.show(10)

+----------+-----+-----------+---------+--------------+---------+---------+
|product_id|month|gross_price|_metadata|read_timestamp|file_name|file_size|
+----------+-----+-----------+---------+--------------+---------+---------+
+----------+-----+-----------+---------+--------------+---------+---------+



### 📌 Step 16: Broadcast Join with Product Master to Attach Surrogate Key
* **Purpose:** Enriches pricing records by joining with `fmcg.silver.products` on natural key `product_id` to attach the conformed surrogate key `product_code`.
* **Logic & Transformations:** Uses `F.broadcast(df_products)` on the small dimension table to prevent cluster-wide shuffle join overhead.
* **Inputs & Dependencies:** `df_silver` pricing and `fmcg.silver.products` master.
* **Outputs & Medallion State:** Enriched DataFrame `df_joined` containing `product_code` alongside monthly gross prices.

In [16]:
# We enrich the silver dataset by performing an optimized broadcast join with products
from pyspark.sql.functions import broadcast
df_products = spark.table(f"{catalog}.{silver_schema}.products")
df_joined = df_silver.join(broadcast(df_products.select("product_id", "product_code")), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")
df_joined.show(5)


[Local Spark Emulation] spark.table adapted: fmcg.silver.products -> silver.products


+----------+------------+-----+-----------+--------------+---------+---------+
|product_id|product_code|month|gross_price|read_timestamp|file_name|file_size|
+----------+------------+-----+-----------+--------------+---------+---------+
+----------+------------+-----+-----------+--------------+---------+---------+



### 📌 Step 17: Persist Enriched Pricing into Silver Delta Table
* **Purpose:** Saves the cleansed, product-joined pricing dataset into Silver Delta Lake with Change Data Feed enabled.
* **Logic & Transformations:** Writes `df_joined` in `overwrite` mode with `mergeSchema=true` and `delta.enableChangeDataFeed=true` to `{catalog}.silver.gross_price`.
* **Inputs & Dependencies:** Enriched DataFrame `df_joined`.
* **Outputs & Medallion State:** Silver Delta table `fmcg.silver.gross_price` committed on storage.

In [17]:
df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true")\
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.silver.gross_price -> silver.gross_price


26/09/17 12:42:06 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


### 📌 Step 18: Query Silver Pricing to Initialize Gold Modeling
* **Purpose:** Reads conformed pricing data from Silver Delta table to begin dimensional modeling.
* **Logic & Transformations:** Queries `fmcg.silver.gross_price` into memory.
* **Inputs & Dependencies:** Silver Delta table `fmcg.silver.gross_price`.
* **Outputs & Medallion State:** DataFrame `df_silver` ready for Gold layer aggregation.

In [18]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.silver.gross_price; -> SELECT * FROM silver.gross_price;


### 📌 Step 19: Project Core Columns for Subsidiary Gold Pricing
* **Purpose:** Selects required dimensional columns (`product_code`, `month`, `gross_price`) for subsidiary monthly pricing.
* **Logic & Transformations:** Calls `df_silver.select('product_code', 'month', 'gross_price')` and previews first 5 rows.
* **Inputs & Dependencies:** Silver DataFrame `df_silver`.
* **Outputs & Medallion State:** Subsidiary Gold DataFrame `df_gold`.

In [19]:
# select only required columns
df_gold = df_silver.select("product_code", "month", "gross_price")
df_gold.show(5)

+------------+-----+-----------+
|product_code|month|gross_price|
+------------+-----+-----------+
+------------+-----+-----------+



### 📌 Step 20: Persist Subsidiary Gold Table (`sb_dim_gross_price`)
* **Purpose:** Writes the monthly subsidiary pricing table `fmcg.gold.sb_dim_gross_price` for transactional margin and revenue calculations.
* **Logic & Transformations:** Writes `df_gold` with `format('delta')` in `overwrite` mode to `{catalog}.gold.sb_dim_gross_price`.
* **Inputs & Dependencies:** Subsidiary DataFrame `df_gold`.
* **Outputs & Medallion State:** Gold Delta table `fmcg.gold.sb_dim_gross_price` persisted on storage.

In [20]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.gold.sb_dim_gross_price -> gold.sb_dim_gross_price


### 📌 Step 21: Query Subsidiary Pricing for Annual Consolidation
* **Purpose:** Reads subsidiary pricing to prepare the annual enterprise reference table `dim_gross_price`.
* **Logic & Transformations:** Calls `spark.table('fmcg.gold.sb_dim_gross_price')` and displays sample records.
* **Inputs & Dependencies:** Subsidiary table `fmcg.gold.sb_dim_gross_price`.
* **Outputs & Medallion State:** DataFrame `df_gold_price` loaded into session.

In [21]:
df_gold_price = spark.table("fmcg.gold.sb_dim_gross_price")
df_gold_price.show(5)

[Local Spark Emulation] spark.table adapted: fmcg.gold.sb_dim_gross_price -> gold.sb_dim_gross_price
+------------+-----+-----------+
|product_code|month|gross_price|
+------------+-----+-----------+
+------------+-----+-----------+



### 📌 Step 22: Annual Price Consolidation & Window Ranking
* **Purpose:** Determines the canonical annual price per `(product_code, year)` by selecting the latest non-zero price available in the calendar year.
* **Logic & Transformations:**
  1. Extracts `year = F.year('month')`.
  2. Flags zero prices: `is_zero = when(gross_price == 0, 1).otherwise(0)`.
  3. Applies `Window.partitionBy('product_code', 'year').orderBy('is_zero', col('month').desc())`.
  4. Filters `row_number == 1` to pick the latest valid price of the year.
* **Inputs & Dependencies:** DataFrame `df_gold_price`.
* **Outputs & Medallion State:** Annualized pricing DataFrame `df_gold_latest_price` with 1 record per `(product_code, year)`.

In [22]:
df_gold_price = (
    df_gold_price
    .withColumn("year", F.year("month"))
    # 0 = non-zero price, 1 = zero price  ➜ non-zero comes first
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)

w = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)


df_gold_latest_price = (
    df_gold_price
      .withColumn("rnk", F.row_number().over(w))
      .filter(F.col("rnk") == 1)
)


### 📌 Step 23: Preview Consolidated Annual Pricing Data
* **Purpose:** Renders an interactive display of consolidated annual prices across products.
* **Logic & Transformations:** Invokes `display(df_gold_latest_price)` for tabular review.
* **Inputs & Dependencies:** Annual pricing DataFrame `df_gold_latest_price`.
* **Outputs & Medallion State:** Tabular display rendered in output.

In [23]:
display(df_gold_latest_price)

+------------+-----+-----------+----+-------+---+
|product_code|month|gross_price|year|is_zero|rnk|
+------------+-----+-----------+----+-------+---+
+------------+-----+-----------+----+-------+---+



### 📌 Step 24: Conformed Schema Projection & Alias Renaming
* **Purpose:** Renames `gross_price` to conformed enterprise column `price_inr` and projects standard parent schema.
* **Logic & Transformations:** Renames column via `.withColumnRenamed('gross_price', 'price_inr')` and projects `(product_code, price_inr, year)`.
* **Inputs & Dependencies:** DataFrame `df_gold_latest_price`.
* **Outputs & Medallion State:** Final parent pricing DataFrame structured according to enterprise data dictionary.

In [24]:
## Take required cols

df_gold_latest_price = df_gold_latest_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

# change year to string
df_gold_latest_price = df_gold_latest_price.withColumn("year", F.col("year").cast("string"))

df_gold_latest_price.show(5)

+------------+---------+----+
|product_code|price_inr|year|
+------------+---------+----+
+------------+---------+----+



### 📌 Step 25: Verify Parent Pricing Schema
* **Purpose:** Confirms data types and nullability of the final parent pricing dataset before merging.
* **Logic & Transformations:** Calls `df_gold_latest_price.printSchema()` to display schema tree.
* **Inputs & Dependencies:** DataFrame `df_gold_latest_price`.
* **Outputs & Medallion State:** Schema tree logged to output.

In [25]:
df_gold_latest_price.printSchema()

root
 |-- product_code: string (nullable = true)
 |-- price_inr: double (nullable = true)
 |-- year: string (nullable = true)



### 📌 Step 26: Data Quality Validation & SCD Type 1 Upsert into `dim_gross_price`
* **Purpose:** Validates price data quality rules and executes an ACID Delta Lake merge into enterprise parent table `fmcg.gold.dim_gross_price`.
* **Logic & Transformations:**
  1. Runs automated quality checks (`PRICING_QUALITY_CHECKS`) ensuring prices are non-null and positive.
  2. Executes `merge()` on composite key `target.product_code = source.product_code AND target.year = source.year`.
  3. Updates matching prices in place; inserts new annual product prices.
* **Inputs & Dependencies:** Conformed DataFrame and target Delta table `fmcg.gold.dim_gross_price`.
* **Outputs & Medallion State:** Enterprise table `fmcg.gold.dim_gross_price` updated with validated annual pricing index.

In [26]:
# Data Quality gate before Gold merge
run_quality_checks(df_gold_latest_price, PRICING_QUALITY_CHECKS, "sb_dim_gross_price")

# Merge into parent dim_gross_price preserving multi-year price history
target_table = f"{catalog}.{gold_schema}.dim_gross_price"
delta_table = DeltaTable.forName(spark, target_table)

delta_table.alias("target").merge(
    source=df_gold_latest_price.alias("source"),
    condition="target.product_code = source.product_code AND target.year = source.year"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()
print(f"Successfully upserted into {target_table} with multi-year preservation.")


[Local Spark Emulation] DeltaTable.forName adapted: fmcg.gold.dim_gross_price -> gold.dim_gross_price


26/09/17 12:42:18 WARN CreateNamespaceExec: Namespace gold was created concurrently. Ignoring.


Successfully upserted into fmcg.gold.dim_gross_price with multi-year preservation.


26/09/17 12:42:21 WARN MapPartitionsRDD: RDD 384 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
